In [33]:
%load_ext autoreload
%autoreload 2

# Define autroreload so that it doesn't cause pain in the ass when we change the functions and run this notebook

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
import sys
from pathlib import Path
project_root = Path.cwd().resolve().parents[2]
sys.path.append(str(project_root))

print(project_root)

from defs.diffusion.diffusion import *
from defs.diffusion.epsilon import *
from defs.diffusion.training_defs import *
from defs.diffusion.loss import *
from defs.diffusion.noise_scheduling import *
from defs.diffusion.noise_sampling import *

import torch.optim as optim

C:\SenkDosya\Projects\FINCH-Science_SyntheticData


In [35]:
# Predefine all the necessary inputs of the training function

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Loss:
loss = loss_mse_sam(0.001)

# Epsilon:
cfg_model = {
    'time_embed': {
        'hidden_dim': 48,
        'hidden_n': 2
    },
    'ab_embed': {
        'hidden_dim': 64,
        'hidden_n': 2,
        'ab_dim': 3
    },
    'denoiser': {
        'hidden_dim': 128,
        'hidden_n': 4,
        'spec_dim': 81
    }
}
epsilon= Epsilon_MLP(cfg_model= cfg_model)
epsilon = epsilon.to(device)

# Scheduler:
scheduler = CosSchedule(2000)

# T sampler:
t_sampler = NormalSampling(scheduler=scheduler, t_min=0, containment_percentage=0.995)

# DDPM:
cfg_diffusion = {
    't_sampler': t_sampler
}
ddpm = cond_diffusion(epsilon=epsilon, scheduler=scheduler, ddpm_dict=cfg_diffusion)

# Optimizer:
optimizer = optim.Adam(epsilon.parameters(), lr=1e-2)

# Data handle:
data_handle = str(project_root)+'\data\simpler_data_rwc.csv'

# cfg_train:
cfg_train = {
    'cfg_loader': {
        'test': 23, 't_batch': 1, 'validate': 4, 'epoch': 50
    },
    'range': [900, 1700],
    'device': device
}


<>:45: SyntaxWarning: invalid escape sequence '\d'
<>:45: SyntaxWarning: invalid escape sequence '\d'


cuda


C:\Users\Ege Artan\AppData\Local\Temp\ipykernel_23072\213922784.py:45: SyntaxWarning: invalid escape sequence '\d'
  data_handle = str(project_root)+'\data\simpler_data_rwc.csv'


In [36]:
print('N of params in epsillon: ', get_n_params(epsilon))

N of params in epsillon:  121217


In [37]:
collector_dict, ds_train, ds_validation, ds_test, ddpm = train_diffusion(
    cfg_train= cfg_train, cond_diffusion= ddpm, loss= loss, optimizer= optimizer, data_handle= data_handle 
)

tensor(1.4004, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.0653, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.7916, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.8542, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.0355, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.8899, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.1339, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.0766, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(1.1783, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.8880, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is total loss
tensor(0.9291, device='cuda:0', dtype=torch.float64, grad_fn=<AddBackward0>)  is

KeyboardInterrupt: 